# Constant Gaussian Curvature with knotted boundary

<!-- More details on this example, can be found in [our paper](https://arxiv.org/abs/2402.14009), Sections 4.1 and A.2. -->

### Imports and setup

In [135]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch import optim
from tqdm.notebook import trange
import k3d
import sys
import os
import time
import copy
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from models.model_architecture import BunnyNet, GeneralNet
from training.optimizers import GaussNewton, GaussNewtonWoodburyBig
from util.surface_sampling import sample_model_surface_binsearch, sample_model_surface_newton 
from util.error_metrics import chamfer_div, compute_distance 
from util.visualization.utils_mesh import get_mesh

torch.manual_seed(0)


device = 'cuda' #if torch.cuda.is_available() else 'cpu'
torch.set_default_device(device)

### True Surface

In [136]:
def sample_wiggly_loop(n_points=1000, dtype=torch.float64, wiggle_freq=2, wiggle_amp=0.5):
    t = torch.linspace(0, 2 * torch.pi, n_points, dtype=dtype)
    x = torch.cos(t)
    y = torch.sin(t)
    z = wiggle_amp * torch.sin(wiggle_freq * t)
    return torch.vstack([x, y, z]).T

def sample_wiggly_ellipse(n_points=1000, dtype=torch.float64, twist=4, amp=0.5):
    t = torch.linspace(0, 2 * torch.pi, n_points, dtype=dtype)
    x = 1.5 * torch.cos(t)
    y = torch.sin(t)
    z = amp * torch.sin(twist * t)
    return torch.vstack([x, y, z]).T

def sample_interlinked_rings(n_points=1000, dtype=torch.float64, sep=1.5):
    half = n_points // 2
    t = torch.linspace(0, 2 * torch.pi, half, dtype=dtype)
    # First ring in x-y plane
    x1 = torch.cos(t)
    y1 = torch.sin(t)
    z1 = torch.zeros_like(t)
    # Second ring in y-z plane, offset in x
    x2 = sep * torch.ones_like(t)
    y2 = torch.cos(t)
    z2 = torch.sin(t)
    ring1 = torch.vstack([x1, y1, z1])
    ring2 = torch.vstack([x2, y2, z2])
    return torch.cat([ring1, ring2], dim=1).T

def sample_costa_inspired_boundary(n_points=1200, dtype=torch.float64):
    third = n_points // 3
    t = torch.linspace(0, 2 * torch.pi, third, dtype=dtype)

    z1 = torch.zeros_like(t)
    y1 = 1.0 * torch.cos(t)
    x1 = 1.0 * torch.sin(t)

    # Left loop in y-z plane (like helicoid end)
    z2 = -1.2 * torch.ones_like(t)
    y2 = 0.4 * torch.cos(t)
    x2 = 0.4 * torch.sin(t)

    # Right loop in y-z plane
    z3 = 1.2 * torch.ones_like(t)
    y3 = 0.4 * torch.cos(t)
    x3 = 0.4 * torch.sin(t)

    loop1 = torch.vstack([x1, y1, z1])
    loop2 = torch.vstack([x2, y2, z2])
    loop3 = torch.vstack([x3, y3, z3])
    return torch.cat([loop1, loop2, loop3], dim=1).T


def sample_helix_with_central_axis(n_points=1000, dtype=torch.float64, R=0.5, H=1.5, N_turns=3):
    n_helix = int(0.75 * n_points)
    n_axis = n_points - n_helix

    # Helix: spiral going from z = -H/2 to z = H/2
    t = torch.linspace(0, 2 * torch.pi * N_turns, n_helix, dtype=dtype)
    x_helix = R * torch.cos(t)
    y_helix = R * torch.sin(t)
    z_helix = H * (t / t[-1] - 0.5)  # shifts range to [-H/2, H/2]

    # Vertical line (z-axis)
    x_axis = torch.zeros(n_axis, dtype=dtype)
    y_axis = torch.zeros(n_axis, dtype=dtype)
    z_axis = torch.linspace(H / 2, -H / 2, n_axis, dtype=dtype)

    x = torch.cat([x_helix, x_axis])
    y = torch.cat([y_helix, y_axis])
    z = torch.cat([z_helix, z_axis])

    return torch.vstack([x, y, z]).T






pts_boundary = sample_helix_with_central_axis(n_points=1000)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)

fig.display()

Output()

### Pretraining

In [137]:
# Generate random training points
model = GeneralNet(ks=[3, 32, 32, 1])
model = model.double()
# Generate random training points
num_pretrain_samples = 10000
bounds = torch.tensor([[-3,3], [-3,3], [-1,1]], dtype=torch.float64)
pts_pretrain = torch.rand(num_pretrain_samples, 3, dtype=torch.float64) * (bounds[:,1] - bounds[:,0]) + bounds[:,0]


# Define pretraining loss
def pretrain_loss(model, params, pts):
    inputs = pts.to(dtype=torch.float64)
    targets = pts[:, 2].to(dtype=torch.float64)
    preds = model(inputs).squeeze(1)
    return 0.5 * (preds - targets).square().mean()

# Pretraining loop
pretrain_optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_pretrain_iters = 1000

for i in range(num_pretrain_iters):
    pretrain_optimizer.zero_grad()
    loss = pretrain_loss(model, model.params, pts_pretrain)
    loss.backward()
    pretrain_optimizer.step()
    
    if i % 100 == 0:
        print(f"Pretrain Iter {i}: Loss = {loss.item():.6f}")

print("Pretraining completed!")

verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-1, -1, -1], dtype=torch.float64),
    bbox_max=torch.tensor([1, 1, 1], dtype=torch.float64),
    chunks=2
)


fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)
fig.display()


Pretrain Iter 0: Loss = 0.272405
Pretrain Iter 100: Loss = 0.003026
Pretrain Iter 200: Loss = 0.001418
Pretrain Iter 300: Loss = 0.000756
Pretrain Iter 400: Loss = 0.000429
Pretrain Iter 500: Loss = 0.000260
Pretrain Iter 600: Loss = 0.000171
Pretrain Iter 700: Loss = 0.000123
Pretrain Iter 800: Loss = 0.000097
Pretrain Iter 900: Loss = 0.000082
Pretraining completed!


Output()

### Main training loop

In [138]:
pts_eikonal = 2 * torch.rand([1000, 3], dtype=torch.float64) - 1
#pts_eikonal = torch.cat((pts_eikonal, pts_boundary))
pts_surface = pts_boundary

# Gauss-Newton weights:
loss_weights = {"interface": 1.0, "eikonal": 0.001, "mean_curvature": 1.0}
# Adam weights:
# loss_weights = {"interface": 1.0, "eikonal": 0.1, "gauss_curvature": 1.0}


config = {
    "pts_boundary": pts_boundary,
    "pts_eikonal": pts_eikonal,
    "pts_surface": pts_surface,
    "loss_weights": loss_weights,
    "regularization": 1e-6,
}

model = model.double()

In [139]:
model = model.double()
params = model.params
# Gauss-Newton:
optimizer = GaussNewton(model, lr=1e-1, config=config)
# Adam:
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for i in (pbar:=trange(10000)):
    optimizer.zero_grad()
    loss_interface  = 0.5*model.f(params, pts_boundary).square().mean()

    loss_eikonal  = 0.5*model.r_eikonal(params, pts_eikonal).squeeze(1).square().mean()

    pts_surface = sample_model_surface_binsearch(model, pts_boundary, bound_limit=1)
    pts_surface = sample_model_surface_newton(model, pts_surface)
    config["pts_surface"] = pts_surface
    optim.config = config
    
    loss_mean_curvature = 0.5*model.r_mean_curvature(params, pts_surface).squeeze(1).square().mean()
       
    loss = loss_weights["interface"] * loss_interface + loss_weights["eikonal"] * loss_eikonal + loss_weights["mean_curvature"] * loss_mean_curvature
    loss.backward()
    pbar.set_description(f"interface: {loss_interface.item():.2e} "
                            f"eikonal: {loss_eikonal.item():.2e} "
                            f"mean curvature: {loss_mean_curvature.item():.2e} "
                            f"{len(pts_surface)}"
                            )
    optimizer.step()

  0%|          | 0/10000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [140]:
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-2, -2, -2], dtype=torch.float64),
    bbox_max=torch.tensor([2, 2, 2], dtype=torch.float64),
    chunks=2
)


fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)
fig.display()

Output()

### Visualize the result

In [103]:
verts, faces = get_mesh(
    model.float(), N=256, device=device,
    bbox_min=torch.tensor([-2, -2, -1.25], dtype=torch.float64),
    bbox_max=torch.tensor([2, 2, 1.25], dtype=torch.float64),
    chunks=2
)

model.double()
mean_curvatures = model.r_mean_curvature(params, torch.tensor(verts, dtype=torch.float64)).squeeze(1).abs()

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)

color_map = k3d.basic_color_maps.Jet
color_range = [0, 0.005]

fig += k3d.mesh(
    verts, faces, 
    attribute=mean_curvatures.cpu().detach().numpy().astype(np.float32),
    color_range=color_range,
    color_map=color_map,
    side='double',
    flat_shading=False
)

fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)

fig.display()

Output()

### LBFGS